In [1]:
!pip install -q beir rank-bm25 scikit-learn

In [2]:
import os, hashlib, json
from beir import util
from beir.datasets.data_loader import GenericDataLoader

url = "https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/cqadupstack.zip"
out_dir = "datasets"
data_path = util.download_and_unzip(url, out_dir)
unix_path = os.path.join(data_path, "unix")

corpus, queries, qrels = GenericDataLoader(data_folder=unix_path).load(split="test")

print(f"Total corpus docs: {len(corpus)}")
print(f"Total queries: {len(queries)}")
print(f"Queries with >=1 gold doc in qrels: {len(qrels)}")

/usr/local/lib/python3.13/dist-packages/beir/util.py:11: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm


datasets/cqadupstack.zip:   0%|          | 0.00/4.98G [00:00<?, ?iB/s]

  0%|          | 0/47382 [00:00<?, ?it/s]

Total corpus docs: 47382
Total queries: 1072
Queries with >=1 gold doc in qrels: 1072


In [3]:
import re
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# 1) Check exact duplicate query texts
q_texts = [queries[qid].strip().lower() for qid in qrels.keys()]
print(f"Unique query strings: {len(set(q_texts))} / {len(q_texts)}")

# 2) Define held-out topic keywords (e.g., systemd / iptables / zsh)
HELD_OUT_PATTERN = re.compile(r"\b(systemd|systemctl|iptables|ufw|zsh)\b", re.I)

held_out_doc_ids = {
    doc_id for doc_id, doc in corpus.items()
    if HELD_OUT_PATTERN.search(doc.get("title", "") + " " + doc.get("text", ""))
}

# Indexable corpus = docs NOT in held-out topics
active_corpus_ids = set(corpus.keys()) - held_out_doc_ids

# Classify each query as answerable vs. unanswerable (all its gold docs were removed)
answerable_qids, unanswerable_qids = [], []
for qid, rel_docs in qrels.items():
    gold_ids = {did for did, score in rel_docs.items() if score > 0}
    if not gold_ids:
        continue
    if gold_ids.isdisjoint(active_corpus_ids):
        unanswerable_qids.append(qid)
    elif gold_ids.issubset(active_corpus_ids):
        answerable_qids.append(qid)
    # (Queries split across both are dropped to prevent partial-answer ambiguity)

print(f"Active corpus docs: {len(active_corpus_ids)} (Held-out: {len(held_out_doc_ids)})")
print(f"Answerable queries: {len(answerable_qids)}")
print(f"Unanswerable (ABSTAIN) queries: {len(unanswerable_qids)}")

Unique query strings: 1072 / 1072
Active corpus docs: 45617 (Held-out: 1765)
Answerable queries: 1030
Unanswerable (ABSTAIN) queries: 27


In [4]:
rng = np.random.default_rng(42)

# Example: sample a balanced ~25-30% unanswerable eval pool, then split 50/50 into val & frozen test
# (Adjust sample size if needed, e.g., 400 answerable + 150 unanswerable)
ans_arr = np.array(sorted(answerable_qids))
unans_arr = np.array(sorted(unanswerable_qids))
rng.shuffle(ans_arr)
rng.shuffle(unans_arr)

mid_a, mid_u = len(ans_arr) // 2, len(unans_arr) // 2
val_ids = {"answerable": ans_arr[:mid_a].tolist(), "unanswerable": unans_arr[:mid_u].tolist()}
test_ids = {"answerable": ans_arr[mid_a:].tolist(), "unanswerable": unans_arr[mid_u:].tolist()}

os.makedirs("data_splits", exist_ok=True)
with open("data_splits/val_ids.json", "w") as f:
    json.dump(val_ids, f, indent=2)
with open("data_splits/test_ids_frozen.json", "w") as f:
    json.dump(test_ids, f, indent=2)

test_hash = hashlib.sha256(open("data_splits/test_ids_frozen.json", "rb").read()).hexdigest()
print("Frozen Test SHA-256:", test_hash)

Frozen Test SHA-256: 681b385901090a3306fb29e6a73a1f1bfca27483d5e29daa9e40a477478b5e4a


In [5]:
import re

topic_groups = {
    "networking_fw": r"\b(ssh|sshd|scp|sftp|iptables|ufw|nmap|tcpdump)\b",
    "pkg_managers":  r"\b(apt|apt-get|dpkg|yum|rpm|pacman|brew)\b",
    "editors":       r"\b(vim|vi|nvim|emacs|nano)\b",
    "version_ctrl":  r"\b(git|svn|mercurial)\b",
    "scheduling":    r"\b(cron|crontab|systemd|systemctl|init\.d)\b",
    "shells_zsh":    r"\b(zsh|csh|tcsh|fish)\b",
}

def count_split(pattern):
    rx = re.compile(pattern, re.I)
    held_out = {did for did, d in corpus.items() if rx.search(d.get("title", "") + " " + d.get("text", ""))}
    active = set(corpus.keys()) - held_out
    ans, unans = 0, 0
    for qid, rel in qrels.items():
        gold = {did for did, s in rel.items() if s > 0}
        if not gold:
            continue
        if gold.isdisjoint(active):
            unans += 1
        elif gold.issubset(active):
            ans += 1
    return len(held_out), len(active), ans, unans

for name, pat in topic_groups.items():
    ho, act, a, u = count_split(pat)
    print(f"{name:15s} -> held_out_docs={ho:5d} | ans={a:4d} | unans={u:3d}")

networking_fw   -> held_out_docs= 4162 | ans= 954 | unans= 82
pkg_managers    -> held_out_docs= 3025 | ans=1007 | unans= 49
editors         -> held_out_docs= 1917 | ans=1033 | unans= 29
version_ctrl    -> held_out_docs=  825 | ans=1049 | unans= 12
scheduling      -> held_out_docs= 1642 | ans=1037 | unans= 23
shells_zsh      -> held_out_docs=  797 | ans=1038 | unans= 19


In [6]:
import os, re, json, hashlib
import numpy as np

# Combine all 6 held-out Unix support topics
HELD_OUT_PATTERN = re.compile(
    r"\b("
    r"ssh|sshd|scp|sftp|iptables|ufw|nmap|tcpdump|"
    r"apt|apt-get|dpkg|yum|rpm|pacman|brew|"
    r"vim|vi|nvim|emacs|nano|"
    r"git|svn|mercurial|"
    r"cron|crontab|systemd|systemctl|init\.d|"
    r"zsh|csh|tcsh|fish"
    r")\b",
    re.I,
)

held_out_doc_ids = {
    doc_id for doc_id, doc in corpus.items()
    if HELD_OUT_PATTERN.search(doc.get("title", "") + " " + doc.get("text", ""))
}
active_corpus_ids = set(corpus.keys()) - held_out_doc_ids

answerable_qids, unanswerable_qids = [], []
for qid, rel_docs in qrels.items():
    gold_ids = {did for did, score in rel_docs.items() if score > 0}
    if not gold_ids:
        continue
    if gold_ids.isdisjoint(active_corpus_ids):
        unanswerable_qids.append(qid)
    elif gold_ids.issubset(active_corpus_ids):
        answerable_qids.append(qid)

# Shuffle & split 50/50 into Validation and Frozen Test (seed=42)
rng = np.random.default_rng(42)
ans_arr = np.array(sorted(answerable_qids))
unans_arr = np.array(sorted(unanswerable_qids))
rng.shuffle(ans_arr)
rng.shuffle(unans_arr)

mid_a, mid_u = len(ans_arr) // 2, len(unans_arr) // 2
val_ids = {"answerable": ans_arr[:mid_a].tolist(), "unanswerable": unans_arr[:mid_u].tolist()}
test_ids = {"answerable": ans_arr[mid_a:].tolist(), "unanswerable": unans_arr[mid_u:].tolist()}

os.makedirs("data_splits", exist_ok=True)
with open("data_splits/val_ids.json", "w") as f:
    json.dump(val_ids, f, indent=2)
with open("data_splits/test_ids_frozen.json", "w") as f:
    json.dump(test_ids, f, indent=2)

test_hash = hashlib.sha256(open("data_splits/test_ids_frozen.json", "rb").read()).hexdigest()

print(f"Total corpus docs:     {len(corpus)}")
print(f"Active indexed docs:   {len(active_corpus_ids)}")
print(f"Held-out topic docs:   {len(held_out_doc_ids)}")
print(f"Answerable queries:    {len(answerable_qids)} (Val: {len(val_ids['answerable'])}, Test: {len(test_ids['answerable'])})")
print(f"Unanswerable (ABSTAIN):{len(unanswerable_qids)} (Val: {len(val_ids['unanswerable'])}, Test: {len(test_ids['unanswerable'])})")
print(f"Frozen Test SHA-256:   {test_hash}")

Total corpus docs:     47382
Active indexed docs:   36257
Held-out topic docs:   11125
Answerable queries:    814 (Val: 407, Test: 407)
Unanswerable (ABSTAIN):192 (Val: 96, Test: 96)
Frozen Test SHA-256:   decc77b8a8ef3745a09b2d34bb5b410613638fc720e96dbd9406601a6a6150bb


In [7]:
import shutil
from google.colab import files

shutil.make_archive("data_splits", "zip", "data_splits")
files.download("data_splits.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>